# 🧩 Mini-Lab: Building an LLM API

**Module 10: Production Deployment** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Design** a RESTful API structure for an LLM application
2. **Build** a FastAPI endpoint that accepts prompts and returns LLM responses
3. **Understand** how request validation and response formatting work in FastAPI
4. **Test** your API interactively using FastAPI's built-in docs

## Target Concepts

| Concept | Description |
|---------|-------------|
| API Design | RESTful AI API design — choosing routes, methods, and data contracts |
| FastAPI Basics | Building web APIs with FastAPI — routes, models, and auto-docs |
| Request Handling | Validating and processing incoming API requests with Pydantic |
| Response Formatting | Returning structured, consistent JSON responses |

## Setup

## What Are We Building?

We'll build a small FastAPI application that exposes an LLM as a REST API. The flow:

```
Client → POST /chat → FastAPI validates request → Calls LLM → Returns structured response
```

We'll write the API code into a Python file, then run it. This is how real LLM APIs are built — as standalone server files, not notebook cells.

## Step 1 — API Design

Good API design starts with deciding **what endpoints you need** and **what data flows in and out**.

For an LLM API, a minimal design looks like:

| Method | Endpoint | Purpose |
|--------|----------|---------|
| `GET` | `/health` | Health check — is the server running? |
| `POST` | `/chat` | Send a prompt, get a completion back |

### Request / Response Contracts

**POST /chat — Request Body:**
```json
{
  "message": "What is prompt engineering?",
  "temperature": 0.7
}
```

**POST /chat — Response Body:**
```json
{
  "response": "Prompt engineering is...",
  "model": "gpt-4o-mini",
  "usage": {
    "prompt_tokens": 12,
    "completion_tokens": 45,
    "total_tokens": 57
  }
}
```

Let's build this now.

## Step 2 — Write the FastAPI Application

We'll write the complete API to a file called `llm_api.py`. This file contains:

1. **Pydantic models** for request validation and response formatting
2. **A health endpoint** (`GET /health`)
3. **A chat endpoint** (`POST /chat`) that calls the OpenAI API

In [2]:
%%writefile llm_api.py
"""Minimal LLM API built with FastAPI."""

from fastapi import FastAPI
from pydantic import BaseModel, Field
from openai import OpenAI
from dotenv import load_dotenv

# ── Load environment variables (.env file with OPENAI_API_KEY) ──
load_dotenv()

# ── Initialize FastAPI app ──
app = FastAPI(
    title="LLM Chat API",
    description="A minimal REST API that wraps an LLM.",
    version="0.1.0",
)

# ── Initialize OpenAI client ──
client = OpenAI()

MODEL = "gpt-4o-mini"


# ═══════════════════════════════════════════
# REQUEST & RESPONSE MODELS (Pydantic)
# ═══════════════════════════════════════════

class ChatRequest(BaseModel):
    """What the client sends to us."""
    message: str = Field(..., min_length=1, description="The user's prompt")
    temperature: float = Field(0.7, ge=0.0, le=2.0, description="Sampling temperature")


class UsageInfo(BaseModel):
    """Token usage details."""
    prompt_tokens: int
    completion_tokens: int
    total_tokens: int


class ChatResponse(BaseModel):
    """What we send back to the client."""
    response: str
    model: str
    usage: UsageInfo


# ═══════════════════════════════════════════
# ENDPOINTS
# ═══════════════════════════════════════════

@app.get("/health")
def health_check():
    """Simple health check — returns OK if the server is running."""
    return {"status": "healthy"}


@app.post("/chat", response_model=ChatResponse)
def chat(request: ChatRequest):
    """Send a message to the LLM and get a response."""
    completion = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": request.message}],
        temperature=request.temperature,
    )

    return ChatResponse(
        response=completion.choices[0].message.content,
        model=completion.model,
        usage=UsageInfo(
            prompt_tokens=completion.usage.prompt_tokens,
            completion_tokens=completion.usage.completion_tokens,
            total_tokens=completion.usage.total_tokens,
        ),
    )

Writing llm_api.py


## Step 3 — Understand the Key Pieces

Let's walk through what we just wrote.

### 🔹 FastAPI Basics

```python
app = FastAPI(title="LLM Chat API", ...)
```

This creates the application. FastAPI automatically generates:
- **Interactive docs** at `http://localhost:8000/docs` (Swagger UI)
- **Alternative docs** at `http://localhost:8000/redoc`
- **OpenAPI schema** at `http://localhost:8000/openapi.json`

### 🔹 Request Handling (Pydantic Validation)

```python
class ChatRequest(BaseModel):
    message: str = Field(..., min_length=1)
    temperature: float = Field(0.7, ge=0.0, le=2.0)
```

Pydantic validates every incoming request **automatically**:
- `message` must be a non-empty string (the `...` means it's required)
- `temperature` defaults to `0.7` and must be between 0.0 and 2.0
- If validation fails, FastAPI returns a `422 Unprocessable Entity` with details

### 🔹 Response Formatting

```python
@app.post("/chat", response_model=ChatResponse)
```

Setting `response_model` does two things:
1. **Documents** the response shape in the auto-generated docs
2. **Validates/serializes** the output — only fields defined in `ChatResponse` are returned

## Step 4 — Run the Server

Run the following cell to start the API server. It will run in the background so you can keep using the notebook.

> **Note:** The server runs on `http://localhost:8000`. You can open `http://localhost:8000/docs` in your browser to see the interactive Swagger UI.

In [3]:
import subprocess
import sys
import time

# Start uvicorn as a background process
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "llm_api:app", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# Give the server a moment to start
time.sleep(3)
print(f"✅ Server started (PID: {server.pid})")
print("📖 Interactive docs: http://localhost:8000/docs")

✅ Server started (PID: 51736)
📖 Interactive docs: http://localhost:8000/docs


## Step 5 — Test the API

Now let's call our API as a client would, using Python's `requests` library.

### Test the Health Check

In [4]:
import requests

resp = requests.get("http://localhost:8000/health")
print(f"Status: {resp.status_code}")
print(f"Body:   {resp.json()}")

Status: 200
Body:   {'status': 'healthy'}


### Test the Chat Endpoint

In [5]:
from IPython.display import display, Markdown
def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

resp = requests.post(
    "http://localhost:8000/chat",
    json={"message": "Explain what an API is in two sentences.", "temperature": 0.3},
)

data = resp.json()
print(f"Status: {resp.status_code}")
print(f"Model:  {data['model']}")
print(f"Usage:  {data['usage']}")
print()
md(data["response"])

Status: 200
Model:  gpt-4o-mini-2024-07-18
Usage:  {'prompt_tokens': 16, 'completion_tokens': 54, 'total_tokens': 70}



An API, or Application Programming Interface, is a set of rules and protocols that allows different software applications to communicate and interact with each other. It defines the methods and data formats that applications can use to request and exchange information, enabling seamless integration and functionality across various systems.

### Test Request Validation

What happens when we send an invalid request? FastAPI + Pydantic handle it automatically.

In [6]:
import json

# Missing required field "message"
resp = requests.post("http://localhost:8000/chat", json={"temperature": 0.5})
print(f"Status: {resp.status_code}")
print(json.dumps(resp.json(), indent=2))

Status: 422
{
  "detail": [
    {
      "type": "missing",
      "loc": [
        "body",
        "message"
      ],
      "msg": "Field required",
      "input": {
        "temperature": 0.5
      }
    }
  ]
}


In [7]:
# Temperature out of range
resp = requests.post(
    "http://localhost:8000/chat",
    json={"message": "Hello", "temperature": 5.0},
)
print(f"Status: {resp.status_code}")
print(json.dumps(resp.json(), indent=2))

Status: 422
{
  "detail": [
    {
      "type": "less_than_equal",
      "loc": [
        "body",
        "temperature"
      ],
      "msg": "Input should be less than or equal to 2",
      "input": 5.0,
      "ctx": {
        "le": 2.0
      }
    }
  ]
}


Notice how FastAPI returns a **422 status code** with a detailed error message — we didn't write any validation logic by hand. Pydantic did it all from our `Field(...)` declarations.

## Step 6 — Check the Auto-Generated Docs

Open **http://localhost:8000/docs** in your browser to see the interactive Swagger UI. You can:

- See all endpoints and their request/response schemas
- Click "Try it out" to make live API calls from the browser
- See the exact Pydantic models rendered as JSON Schema

This is one of FastAPI's biggest advantages — **your API is self-documenting**.

## Cleanup

Stop the server and remove the generated file.

In [8]:
import os

# Stop the server
server.terminate()
server.wait()
print("🛑 Server stopped.")

# Clean up generated file
if os.path.exists("llm_api.py"):
    os.remove("llm_api.py")
    print("🧹 Removed llm_api.py")

🛑 Server stopped.
🧹 Removed llm_api.py


## 🎯 Summary

### Key Takeaways

1. **API Design** — a minimal LLM API needs a health check endpoint (`GET /health`) and a chat endpoint (`POST /chat`) with clear request/response contracts
2. **FastAPI Basics** — `FastAPI()` creates an app with auto-generated interactive docs at `/docs`, making your API self-documenting
3. **Request Handling** — Pydantic models validate incoming data automatically; invalid requests get a `422` response with detailed errors, with zero manual validation code
4. **Response Formatting** — setting `response_model` on an endpoint ensures consistent, typed JSON responses and documents the output schema